[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/prefect-certified/notebooks/day-13-review-advanced.ipynb#scrollTo=a1b2c3d4)

---
# Day 13 · Review — Advanced Patterns and Real-World Gotchas
**certified-journeys / prefect-certified** · Day 13 · Review

> **Goal for today:** Consolidate everything from Days 1–12 by diagnosing common production failures, hardening tasks with idempotency and retries, comparing deployment strategies, and writing a runbook you can reach for when things break.


In [ ]:
%pip install -q prefect==3.* httpx pandas


## Step 1 · Idempotency — Same inputs, same outputs, no side effects

An **idempotent** task produces the same result every time it is called with the same inputs, and does not accumulate side effects on re-runs.
This matters most when:
- A flow crashes mid-run and must be retried from scratch.
- You re-run the same scheduled execution after fixing a bug.
- You are backfilling historical data.

### Non-idempotent vs idempotent write

| Pattern | Problem | Fix |
|---|---|---|
| `df.to_csv('out.csv', mode='a')` | Appends duplicates on re-run | Use `mode='w'` or check existence |
| `INSERT INTO table …` | Inserts duplicates | Use `INSERT OR REPLACE` / upsert |
| `requests.post(url, data)` | Sends duplicate events | Add idempotency key header |

Prefect's **`task_input_hash`** cache key helper makes read-compute tasks idempotent automatically.


In [ ]:
import os
import hashlib
import json
from pathlib import Path
from prefect import flow, task
from prefect.tasks import task_input_hash
from datetime import timedelta

# ── Idempotent file-write task ──────────────────────────────────────────────
# Writing to a deterministic path makes this safe to re-run: the second run
# simply overwrites the same file rather than appending.

@task(cache_key_fn=task_input_hash, cache_expiration=timedelta(hours=1))
def compute_summary(records: list[dict]) -> dict:
    """Pure function: same records always produce the same summary."""
    total = sum(r["value"] for r in records)
    return {"count": len(records), "total": total, "mean": total / len(records)}


@task
def write_result(summary: dict, output_path: str) -> None:
    """Idempotent write: overwrites existing file on re-run."""
    path = Path(output_path)
    path.parent.mkdir(parents=True, exist_ok=True)  # safe to call repeatedly
    path.write_text(json.dumps(summary, indent=2))
    print(f"Written: {path}  (size={path.stat().st_size} bytes)")


@flow(name="idempotency-demo")
def idempotent_pipeline(run_id: str = "run-001"):
    records = [
        {"id": 1, "value": 10},
        {"id": 2, "value": 20},
        {"id": 3, "value": 30},
    ]
    # The cache means the second call skips re-computation
    summary = compute_summary(records)
    write_result(summary, f"/tmp/prefect-review/{run_id}/summary.json")
    return summary


# Run twice — the second call hits the cache and skips compute_summary
result1 = idempotent_pipeline(run_id="run-001")
result2 = idempotent_pipeline(run_id="run-001")
print("Both results equal:", result1 == result2)


### What just happened?

- **`task_input_hash`** hashes every argument to `compute_summary` and stores the result in Prefect's cache backend.
- The second flow run with identical `records` found the cached result and **returned it without executing the function body**.
- **`write_result` is not cached** — it is a side effect. But using `mode='w'` (or `path.write_text`) makes it safe to re-run: you overwrite, not accumulate.
- **Key insight:** caching handles the _read/compute_ side; deterministic output paths handle the _write_ side.


## Step 2 · Retries and Timeouts on External API Tasks

Every task that calls an external service should have:

| Parameter | Why |
|---|---|
| `retries=3` | Networks fail; transient 5xx errors are common |
| `retry_delay_seconds=10` | Avoid hammering a struggling service |
| `timeout_seconds=30` | Prevent a hung connection from blocking the worker |

Prefect's `@task` decorator accepts all three.  
For HTTP specifically, also set `httpx.Client(timeout=…)` or `requests.get(timeout=…)` — network timeouts and Prefect task timeouts serve different layers.

### Retry jitter
Retry storms can overwhelm a recovering service. Use `exponential_backoff` from `prefect.tasks` to multiply the delay on each attempt.


In [ ]:
import httpx
import random
from prefect import flow, task
from prefect.tasks import exponential_backoff

# ── Mock HTTP task with retries, delay, and timeout ────────────────────────

CALL_COUNT = 0  # module-level counter to simulate transient failures


@task(
    retries=3,
    retry_delay_seconds=exponential_backoff(backoff_factor=2),  # 2s, 4s, 8s
    timeout_seconds=15,
    log_prints=True,
)
def fetch_post(post_id: int) -> dict:
    """Fetch a single post from JSONPlaceholder (free mock REST API)."""
    global CALL_COUNT
    CALL_COUNT += 1
    print(f"Attempt #{CALL_COUNT} for post_id={post_id}")

    # httpx timeout is the *network* timeout; Prefect timeout_seconds is the
    # *task-level* wall-clock limit — both are needed.
    with httpx.Client(timeout=10.0) as client:
        response = client.get(
            f"https://jsonplaceholder.typicode.com/posts/{post_id}"
        )
        response.raise_for_status()  # raises on 4xx/5xx — triggers retry
        return response.json()


@flow(name="retry-demo", log_prints=True)
def fetch_posts(post_ids: list[int]):
    results = []
    for pid in post_ids:
        data = fetch_post(pid)
        print(f"  Got: {data['title'][:40]}…")
        results.append(data)
    return results


posts = fetch_posts([1, 2, 3])
print(f"\nFetched {len(posts)} posts successfully.")


### What just happened?

- **`exponential_backoff(backoff_factor=2)`** returns a list `[2, 4, 8]` of wait seconds for each retry attempt.
- `raise_for_status()` converts HTTP error codes into Python exceptions — Prefect sees these and triggers the retry logic.
- **Two timeout layers:** `httpx` cuts the TCP connection if the server is slow to respond; Prefect's `timeout_seconds` kills the task process if it exceeds the wall-clock limit.
- **Key insight:** without retries, a single transient DNS hiccup fails the entire flow run. Three retries covers 99.9% of transient failures observed in practice.


## Step 3 · Deployment Strategies Compared

Prefect offers three primary ways to create and manage deployments. Understanding the trade-offs helps you pick the right tool for each context.

| Strategy | Command / API | Best for | Limitations |
|---|---|---|---|
| **Python API** | `flow.deploy(…)` | Quick local deploys, scripted setup | Imperative; hard to diff in git |
| **`prefect.yaml`** | `prefect deploy` CLI | Team repos, GitOps workflows | Requires Prefect Cloud or server |
| **CI/CD dispatch** | `prefect deployment run` in GitHub Actions / GitLab CI | Production promotion gates, multi-env | More infra to maintain |

### `prefect.yaml` anatomy

```yaml
# prefect.yaml
name: etl-pipeline
prefect-version: "3.*"

build: null
push: null

pull:
  - prefect.deployments.steps.git_clone:
      repository: https://github.com/your-org/your-repo
      branch: main

deployments:
  - name: daily-etl
    entrypoint: flows/etl.py:etl_flow
    work_pool:
      name: process-pool
    schedules:
      - cron: "0 6 * * *"       # every day at 06:00 UTC
        timezone: UTC
    parameters:
      env: production
```

### CI/CD dispatch pattern

```yaml
# .github/workflows/deploy.yml (excerpt)
- name: Trigger Prefect run
  run: |
    prefect deployment run 'etl-flow/daily-etl' \
      --param env=staging
  env:
    PREFECT_API_KEY: ${{ secrets.PREFECT_API_KEY }}
    PREFECT_API_URL: ${{ secrets.PREFECT_API_URL }}
```


In [ ]:
# ── Deployment strategies: Python API demo (local, no server needed) ───────
# We compare the three approaches programmatically and show what each
# deploy() call would look like if a Prefect server were available.

from prefect import flow, task


@task(log_prints=True)
def extract(source: str) -> list[dict]:
    print(f"Extracting from {source}")
    return [{"id": i, "value": i * 10} for i in range(1, 6)]


@task(log_prints=True)
def transform(records: list[dict]) -> list[dict]:
    return [{**r, "value_doubled": r["value"] * 2} for r in records]


@flow(name="strategy-demo", log_prints=True)
def etl_flow(source: str = "local", env: str = "dev"):
    print(f"Running in env={env}")
    raw = extract(source)
    transformed = transform(raw)
    print(f"Processed {len(transformed)} records")
    return transformed


# ── Strategy 1: Python API ──────────────────────────────────────────────────
# When a server is available:
#
#   deployment_id = await etl_flow.deploy(
#       name="daily-etl",
#       work_pool_name="process-pool",
#       cron="0 6 * * *",
#       parameters={"env": "production"},
#   )

# ── Strategy 2: prefect.yaml (declarative) ─────────────────────────────────
# CLI: prefect deploy --name daily-etl
# The yaml file in the repo root drives everything — source of truth is git.

# ── Strategy 3: CI/CD dispatch ─────────────────────────────────────────────
# Shell: prefect deployment run 'strategy-demo/daily-etl' --param env=staging

# For this notebook we run the flow directly to verify the logic works
result = etl_flow(source="test-source", env="dev")
print("\nSample output:", result[:2])


# ── Quick comparison summary ───────────────────────────────────────────────
strategies = {
    "Python API": {"declarative": False, "git_diffable": False, "ci_native": False},
    "prefect.yaml": {"declarative": True,  "git_diffable": True,  "ci_native": True},
    "CI/CD dispatch": {"declarative": False, "git_diffable": True,  "ci_native": True},
}

print("\nDeployment strategy comparison:")
print(f"{'Strategy':<20} {'Declarative':<14} {'Git-diffable':<14} {'CI-native'}")
print("-" * 60)
for name, props in strategies.items():
    print(f"{name:<20} {str(props['declarative']):<14} {str(props['git_diffable']):<14} {props['ci_native']}")


### What just happened?

- The flow runs cleanly in all three deployment modes — the *execution logic* does not change; only the *scheduling mechanism* differs.
- **Python API** is fastest for experiments but the deploy config lives only in your script, not in version control.
- **`prefect.yaml`** is the recommended production pattern: the entire deployment spec is committed, reviewed, and diffed like any other code.
- **Key insight:** choose `prefect.yaml` for team repos; add CI/CD dispatch on top for environment promotion gates.


## Step 4 · Flow Run State Tracing and Root-Cause Analysis

Prefect's state model distinguishes several terminal states. Understanding them speeds up incident response.

| State | Meaning | Common cause |
|---|---|---|
| `COMPLETED` | All tasks and flow returned successfully | — |
| `FAILED` | A task or the flow raised an exception | Bug in your code, upstream API down |
| `CRASHED` | The infrastructure died mid-run | OOM kill, worker restart, timeout |
| `CANCELLED` | Manually cancelled or auto-cancelled by an automation | Operator action |
| `PENDING` | Waiting for a worker to pick up the run | No running worker for the work pool |

### Tracing a CRASHED run

1. Open the Prefect UI → **Flow Runs** → filter `state:CRASHED`.
2. Click the run → **Logs** tab → look for the last log line before the crash.
3. Check the worker host: `journalctl -u prefect-worker -n 100` or your container logs.
4. Common culprit: OOM — check `dmesg | grep -i oom`.

### FAILED vs CRASHED distinction

```
FAILED  → Python exception bubbled up  → fix your code
CRASHED → process was killed externally → fix your infra
```


In [ ]:
# ── Demonstrate state introspection using the Python client ───────────────
# In production you would query your Prefect server; here we construct
# mock state objects to show the pattern.

from prefect.states import Completed, Failed, Crashed
from prefect.results import UnpersistedResult


def classify_state(state) -> str:
    """Return a human-readable diagnosis for a flow run state."""
    if state.is_completed():
        return "✅ Success — no action needed"
    elif state.is_failed():
        return "🔴 FAILED — check task logs for Python traceback"
    elif state.is_crashed():
        return "💥 CRASHED — check worker host for OOM or process kill"
    elif state.is_cancelled():
        return "⏹ CANCELLED — manual action or automation rule"
    elif state.is_pending():
        return "⏳ PENDING — no worker polling this work pool"
    else:
        return f"❓ Unknown state: {state.type}"


# Simulate the three most-encountered terminal states
mock_states = [
    Completed(message="All tasks succeeded"),
    Failed(message="ValueError: invalid date format in row 42"),
    Crashed(message="Process killed with signal 9 (OOM)"),
]

for s in mock_states:
    print(f"{s.type.value:<12} → {classify_state(s)}")


# ── Programmatic run history query (requires server) ─────────────────────
# from prefect.client.orchestration import get_client
# from prefect.client.schemas.filters import FlowRunFilter
# from prefect.client.schemas.objects import StateType
#
# async with get_client() as client:
#     runs = await client.read_flow_runs(
#         flow_run_filter=FlowRunFilter(
#             state={"type": {"any_": [StateType.FAILED, StateType.CRASHED]}}
#         ),
#         limit=10,
#     )
#     for run in runs:
#         print(run.name, run.state_type, run.start_time)


### What just happened?

- Prefect state objects expose `.is_completed()`, `.is_failed()`, `.is_crashed()` etc. — use these in monitoring scripts to route alerts correctly.
- **`FAILED` is your code; `CRASHED` is your infra** — keeping this distinction in mind prevents wasted debugging time.
- The commented-out block shows how you'd query the server programmatically to build a custom alerting pipeline.
- **Key insight:** build a nightly audit script that surfaces any CRASHED runs — these often go unnoticed because Prefect cannot send a failure notification if the process died before it could reach the API.


## Step 5 · Cache Key Collisions — A Subtle Production Gotcha

Prefect task caching is keyed on the *serialised function arguments*. Collisions can cause a task to return a **stale result** when you expected fresh data.

### When collisions bite you

| Scenario | Why it's a collision |
|---|---|
| Two tasks with different names but identical arguments | Same hash → one task gets the other's cached output |
| A list arg whose items are mutable dicts | Dict key order can affect hash; use sorted keys |
| A `datetime.now()` arg inside a cached task | Non-deterministic input → cache never hits |
| Changing task code without bumping cache version | Old cached result returned for new logic |

### Best practices

1. Always pair `task_input_hash` with a `cache_expiration` — never cache indefinitely.
2. For data-freshness-sensitive tasks, include a `run_date: date` parameter in the function signature so the cache key changes per day.
3. Use `cache_key_fn=lambda ctx, args: f"{ctx.task_run.task_name}:{task_input_hash(ctx, args)}"` to namespace by task name and prevent cross-task collisions.


In [ ]:
# ── Cache key strategies side-by-side ─────────────────────────────────────
from datetime import date, timedelta
from prefect import flow, task
from prefect.tasks import task_input_hash


# PATTERN 1: Simple hash — fine for pure functions
@task(cache_key_fn=task_input_hash, cache_expiration=timedelta(hours=1))
def fetch_config(config_name: str) -> dict:
    print(f"  [fetch_config] Loading {config_name}")
    return {"name": config_name, "version": 1}


# PATTERN 2: Date-scoped cache — result refreshes every day
def daily_cache_key(context, arguments):
    """Cache key includes today's date → expires naturally at midnight."""
    base = task_input_hash(context, arguments)
    return f"{date.today().isoformat()}:{base}"


@task(cache_key_fn=daily_cache_key)
def fetch_daily_metrics(region: str) -> dict:
    print(f"  [fetch_daily_metrics] Querying {region} for {date.today()}")
    return {"region": region, "date": date.today().isoformat(), "records": 1000}


# PATTERN 3: Task-name-namespaced cache — prevents cross-task collisions
def namespaced_cache_key(context, arguments):
    base = task_input_hash(context, arguments)
    task_name = context.task_run.task_name
    return f"{task_name}:{base}"


@task(cache_key_fn=namespaced_cache_key, cache_expiration=timedelta(hours=6))
def fetch_user_data(user_id: int) -> dict:
    print(f"  [fetch_user_data] Fetching user {user_id}")
    return {"user_id": user_id, "name": f"User{user_id}"}


@flow(name="cache-key-demo", log_prints=True)
def cache_patterns_demo():
    print("--- Pattern 1: simple hash ---")
    c1 = fetch_config("database")
    c2 = fetch_config("database")  # cache hit — no print

    print("--- Pattern 2: daily scoped ---")
    m1 = fetch_daily_metrics("us-east-1")

    print("--- Pattern 3: namespaced ---")
    u1 = fetch_user_data(42)

    return {"config": c1, "metrics": m1, "user": u1}


demo_result = cache_patterns_demo()
print("\nResults:", demo_result)


### What just happened?

- Three distinct caching strategies were demonstrated: simple hash, date-scoped, and task-name-namespaced.
- **`fetch_config("database")`** printed only once — the second call was a cache hit.
- **`daily_cache_key`** incorporates `date.today()` so the result automatically expires at midnight without needing `cache_expiration`.
- **Key insight:** when you have two tasks that accept the same arguments, always namespace your cache keys — otherwise task A's cached result can be silently returned for task B.


## Step 6 · Production Runbook — Six Common Incidents

The following runbook distils the most common production Prefect issues and their remediation steps. Keep a copy of this in your team's incident wiki.


In [ ]:
# ── Generate and print the production runbook ────────────────────────────

RUNBOOK = [
    {
        "incident": "Worker crashes — deployments silently stop running",
        "symptoms": [
            "Scheduled runs stay in PENDING state",
            "No recent COMPLETED runs in the UI",
        ],
        "diagnosis": "prefect worker ls  →  look for workers with stale heartbeat",
        "fix": "Restart worker: systemctl restart prefect-worker  OR  docker restart <worker>",
        "prevention": "Set up Prefect Automation: alert if no flow run completes within 2× schedule interval",
    },
    {
        "incident": "Stale schedule — runs deploy but never execute at the right time",
        "symptoms": [
            "Deployment exists but next_scheduled_start_time is wrong",
            "Runs execute at unexpected intervals",
        ],
        "diagnosis": "prefect deployment inspect <name>  →  check schedules section and timezone",
        "fix": "Re-deploy with explicit timezone: cron='0 6 * * *', timezone='America/New_York'",
        "prevention": "Always specify timezone in cron schedules — never rely on server defaults",
    },
    {
        "incident": "Cache key collision — task returns stale data for a new input",
        "symptoms": [
            "Task logs show no print output (cache hit)",
            "Output is clearly wrong for the given inputs",
        ],
        "diagnosis": "Check Prefect UI: task state shows CACHED  →  inspect cache key in state details",
        "fix": "Expire cache: prefect task cache purge <task-name>  OR  bump cache version in key fn",
        "prevention": "Use namespaced cache keys and always set cache_expiration",
    },
    {
        "incident": "State mismatch — UI says COMPLETED but output file is missing",
        "symptoms": [
            "Flow shows COMPLETED in UI",
            "Downstream pipeline cannot find expected output",
        ],
        "diagnosis": "Inspect task logs for write step — check if write_result task state is also COMPLETED",
        "fix": "Add explicit artifact creation with prefect.artifacts.create_table_artifact() so outputs are tracked",
        "prevention": "Separate compute tasks (cacheable) from side-effect tasks (never cached); assert output exists",
    },
    {
        "incident": "Retry storm — failing task retries hammer a recovering service",
        "symptoms": [
            "Downstream service logs spike of identical requests",
            "Service stays down because Prefect retries overwhelm recovery",
        ],
        "diagnosis": "Check retry_delay_seconds — flat delays cause synchronised retries across concurrent tasks",
        "fix": "Switch to exponential_backoff with jitter: retry_jitter_factor=0.5",
        "prevention": "Standard retry config: retries=3, exponential_backoff(2), retry_jitter_factor=0.5",
    },
    {
        "incident": "Deployment parameter drift — prod and staging run with different configs",
        "symptoms": [
            "Prod run uses wrong bucket/table names",
            "Parameters were changed manually in the UI and not reflected in code",
        ],
        "diagnosis": "prefect deployment inspect <name> | grep parameters  →  compare with prefect.yaml",
        "fix": "Re-deploy from prefect.yaml to reset parameters to their declared values",
        "prevention": "Treat the UI as read-only for parameters; all config changes go through prefect.yaml + CI/CD",
    },
]

print("═" * 70)
print("  PREFECT PRODUCTION RUNBOOK — 6 Common Incidents")
print("═" * 70)

for i, entry in enumerate(RUNBOOK, 1):
    print(f"\n{'─'*70}")
    print(f"#{i}: {entry['incident']}")
    print(f"  Symptoms : {'; '.join(entry['symptoms'])}")
    print(f"  Diagnosis: {entry['diagnosis']}")
    print(f"  Fix      : {entry['fix']}")
    print(f"  Prevent  : {entry['prevention']}")

print(f"\n{'═'*70}")


### What just happened?

- Six production incidents were codified with symptoms, diagnosis steps, immediate fixes, and prevention strategies.
- **The silent worker crash** is the most dangerous — runs queue in PENDING forever and nothing alerts you unless you have an Automation rule.
- **State mismatch** is the subtlest: a COMPLETED state only means Prefect ran successfully, not that your output data was actually written.
- **Key insight:** build alerting for PENDING runs accumulating — a healthy system should never have more than 1–2 runs stuck in PENDING for more than 5 minutes.


In [ ]:
# Challenge: Idempotent upsert task
#
# Implement a task `upsert_records` that:
#   1. Accepts a list of dicts, each with an 'id' key.
#   2. Reads an existing JSON file at `output_path` if it exists.
#   3. Merges incoming records into the existing data, keying on 'id'
#      (new records are added; existing ids are UPDATED, not duplicated).
#   4. Writes the merged result back to `output_path`.
#   5. Is safe to call twice with the same inputs (idempotent).
#
# Then wrap it in a flow and verify idempotency by running the flow twice.

# Your solution here

# Scaffold:
# @task
# def upsert_records(records: list[dict], output_path: str) -> int:
#     existing = {}  # load from output_path if file exists
#     # ... merge logic ...
#     # ... write merged result ...
#     return len(merged)

# @flow
# def upsert_flow(records, path):
#     count = upsert_records(records, path)
#     print(f"Total records after upsert: {count}")


---
## Day 13 key concepts recap

| Concept | What to remember |
|---|---|
| Idempotency | Same inputs → same outputs; use `task_input_hash` for compute, deterministic paths for writes |
| Retries | Always add `retries` + `retry_delay_seconds` to external API tasks; use `exponential_backoff` to avoid retry storms |
| Timeout layers | Set both `httpx` network timeout AND Prefect `timeout_seconds` |
| Deployment strategies | Python API = quick; `prefect.yaml` = team/GitOps; CI/CD dispatch = promotion gates |
| Cache collisions | Namespace keys by task name; always set `cache_expiration` |
| FAILED vs CRASHED | FAILED = Python exception; CRASHED = infra killed the process |
| Silent worker crash | The #1 production gotcha — set an Automation alert for PENDING accumulation |

> **Tip:** The most common production issue — a worker goes down and deployments silently stop running. Set up an automation that alerts you if no runs complete within a given time window.

---
## What's next
**Day 14** → Capstone — build a full production ETL pipeline with scheduling, retries, subflows, and automated failure notifications. Apply every pattern from this review in a single end-to-end project.

Mark Day 13 complete in your [tracker](../index.html).
